# 06 · Catálogo de dados no Unity Catalog

Grava a descrição de cada tabela e de cada coluna das 10 tabelas do MVP como comentário no Unity Catalog:
- `bronze.reclamacoes`;
- `silver.reclamacoes`;
- as 8 tabelas da Gold.

Cada comentário de coluna tem três partes:

| Parte | De onde vem |
|---|---|
| **Descrição** | Escrita com base no dicionário de dados da fonte (Senacon/MJ, v1.1) e nas decisões dos notebooks 03 a 05 |
| **Domínio** | **Medido por este notebook** a cada execução: mínimo e máximo em números e datas; lista de valores quando há até 12 distintos; total de distintos quando há mais; contagem de nulos |
| **Linhagem** | De onde a coluna vem e qual transformação sofreu, conforme o código dos notebooks 02, 04 e 05 |

A última seção lê os comentários de volta do `information_schema` e imprime o catálogo em Markdown para o README.

## 1. Descrições e linhagem (escritas)

In [ ]:
from pyspark.sql import functions as F

RECORTE = "segmentos Bancos, Financeiras e Administradoras de Cartão e Empresas de Pagamento Eletrônico"

CATALOGO = {
    "mvp_reclamacoes.bronze.reclamacoes": {
        "descricao": (
            "Reclamações finalizadas do consumidor.gov.br (Senacon/MJ, licença CC BY), de jan/2021 a ago/2026, "
            "de todos os setores, exatamente como publicadas: as colunas da fonte como texto, com os nomes originais "
            "(column mapping), mais 2 colunas de controle. Campo vazio no CSV = NULL. Origem: 68 arquivos .csv.gz do volume "
            "mvp_reclamacoes.bronze.arquivos, gerados por scripts/baixar_reclamacoes.py. Carga: notebooks/02_bronze."
        ),
        "colunas": {
            "Região": ("Sigla da região geográfica do consumidor reclamante.", "Coluna Região dos CSVs, sem alteração."),
            "UF": ("Sigla do estado do consumidor reclamante.", "Coluna UF dos CSVs, sem alteração."),
            "Cidade": ("Município do consumidor reclamante.", "Coluna Cidade dos CSVs, sem alteração."),
            "Sexo": ("Sigla do sexo do consumidor. O dicionário da fonte documenta F e M; os dados também trazem O.",
                     "Coluna Sexo dos CSVs, sem alteração."),
            "Faixa Etária": ("Faixa etária do consumidor.", "Coluna Faixa Etária dos CSVs, sem alteração."),
            "Data Finalização": ("Data de finalização da reclamação, como texto: aaaa-mm-dd na maioria dos meses e dd/mm/aaaa no arquivo de 2021-12.",
                                 "Coluna Data Finalização dos CSVs, sem alteração."),
            "Tempo Resposta": ("Dias que a empresa levou para responder, como texto; vazio quando a reclamação não foi respondida.",
                               "Coluna Tempo Resposta dos CSVs, sem alteração."),
            "Nome Fantasia": ("Nome pelo qual a empresa reclamada é conhecida no mercado.", "Coluna Nome Fantasia dos CSVs, sem alteração."),
            "Segmento de Mercado": ("Principal segmento de mercado da empresa participante.", "Coluna Segmento de Mercado dos CSVs, sem alteração."),
            "Área": ("Área à qual pertence o assunto da reclamação.", "Coluna Área dos CSVs, sem alteração."),
            "Assunto": ("Assunto (produto ou serviço) objeto da reclamação.", "Coluna Assunto dos CSVs, sem alteração."),
            "Grupo Problema": ("Agrupamento do problema classificado na reclamação.", "Coluna Grupo Problema dos CSVs, sem alteração."),
            "Problema": ("Descrição do problema objeto da reclamação.", "Coluna Problema dos CSVs, sem alteração."),
            "Como Comprou Contratou": ("Meio utilizado para contratar ou adquirir o produto ou serviço reclamado.",
                                       "Coluna Como Comprou Contratou dos CSVs, sem alteração."),
            "Procurou Empresa": ("Resposta do consumidor à pergunta: procurou a empresa para solucionar o problema? (S ou N).",
                                 "Coluna Procurou Empresa dos CSVs, sem alteração."),
            "Respondida": ("Resposta à pergunta: a empresa respondeu a reclamação? (S ou N).", "Coluna Respondida dos CSVs, sem alteração."),
            "Situação": ("Situação da reclamação no sistema.", "Coluna Situação dos CSVs, sem alteração."),
            "Avaliação Reclamação": ("Classificação dada pelo consumidor ao desfecho da reclamação.", "Coluna Avaliação Reclamação dos CSVs, sem alteração."),
            "Nota do Consumidor": ("Nota de 1 a 5 dada pelo consumidor ao atendimento, como texto; vazia se a reclamação não foi avaliada.",
                                   "Coluna Nota do Consumidor dos CSVs, sem alteração."),
            "Interação com Judiciario": ("Coluna publicada só no arquivo de 2025-09 (S ou N); nula nos demais meses. Não consta no dicionário v1.1.",
                                         "Coluna Interação com Judiciario do CSV de 2025-09, sem alteração."),
            "Último Complemento Consumidor": ("Data do último complemento do consumidor, como texto dd/mm/aaaa, publicada só no arquivo de 2025-09; nula nos demais meses. Não consta no dicionário v1.1.",
                                              "Coluna Último Complemento Consumidor do CSV de 2025-09, sem alteração."),
            "_arquivo_origem": ("Coluna de controle: nome do arquivo .csv.gz de onde a linha veio.", "Gerada na carga a partir de _metadata.file_name (notebooks/02_bronze)."),
            "_data_ingestao": ("Coluna de controle: momento da carga na Bronze.", "Gerada na carga com current_timestamp() (notebooks/02_bronze)."),
        },
    },
    "mvp_reclamacoes.silver.reclamacoes": {
        "descricao": (
            "Reclamações de todos os setores, limpas e tipadas: uma linha para cada linha da Bronze, nomes em snake_case, "
            "trim em todo texto, datas e inteiros convertidos, S/N em booleano, de-para de problema, nome de empresa padronizado "
            "e marcação de tempos de resposta inválidos e de linhas repetidas. Nenhuma linha é removida. "
            "Origem: mvp_reclamacoes.bronze.reclamacoes. Transformação: notebooks/04_silver."
        ),
        "colunas": {
            "regiao": ("Sigla da região do consumidor.", "bronze.Região + trim (a fonte publica N e S com espaço no fim)."),
            "uf": ("Sigla do estado do consumidor.", "bronze.UF + trim."),
            "cidade": ("Município do consumidor.", "bronze.Cidade + trim."),
            "sexo": ("Código do sexo do consumidor: F, M ou O. Os rótulos ficam em gold.dim_perfil.", "bronze.Sexo + trim."),
            "faixa_etaria": ("Faixa etária do consumidor.", "bronze.Faixa Etária + trim."),
            "data_finalizacao": ("Data de finalização da reclamação.",
                                 "bronze.Data Finalização + trim, convertida para DATE com try_to_date nos formatos aaaa-mm-dd e dd/mm/aaaa."),
            "tempo_resposta_dias": ("Dias que a empresa levou para responder; nulo se não respondida ou se o valor publicado passava de 365 dias.",
                                    "bronze.Tempo Resposta + trim + try_cast para INT; acima de 365 vira nulo (notebooks/03, H7b)."),
            "tempo_resposta_invalido": ("Verdadeiro quando o tempo publicado passava de 365 dias e foi anulado.",
                                        "Calculada na Silver a partir de bronze.Tempo Resposta."),
            "nome_fantasia_original": ("Nome da empresa exatamente como publicado.", "bronze.Nome Fantasia, sem alteração."),
            "nome_empresa": ("Nome da empresa padronizado: a grafia mais frequente entre as que só diferem em maiúsculas, espaços e acentos.",
                             "bronze.Nome Fantasia + trim + chave normalizada (minúsculas, espaços, sem acento) + grafia mais frequente por chave (notebooks/03, H4)."),
            "segmento_mercado": ("Principal segmento de mercado da empresa.", "bronze.Segmento de Mercado + trim."),
            "area": ("Área à qual pertence o assunto.", "bronze.Área + trim."),
            "assunto": ("Assunto (produto ou serviço) da reclamação.", "bronze.Assunto + trim."),
            "grupo_problema": ("Agrupamento do problema.", "bronze.Grupo Problema + trim."),
            "problema_original": ("Problema exatamente como publicado.", "bronze.Problema, sem alteração."),
            "problema": ("Problema padronizado.",
                         "bronze.Problema + trim + de-para de 6 nomes: travessão trocado por ? pela fonte a partir de 2021-12, erro de digitação do SAC e renomeação do telemarketing para (0303) (notebooks/03, H2b e H2c)."),
            "canal_contratacao": ("Meio de contratação ou aquisição do produto ou serviço.", "bronze.Como Comprou Contratou + trim."),
            "procurou_empresa": ("Se o consumidor procurou a empresa antes de reclamar.", "bronze.Procurou Empresa: S → verdadeiro, N → falso."),
            "respondida": ("Se a empresa respondeu a reclamação.", "bronze.Respondida: S → verdadeiro, N → falso."),
            "situacao": ("Situação da reclamação no sistema. Não é usada na Gold, porque diverge da avaliação em alguns registros (notebooks/03, H7c).",
                         "bronze.Situação + trim."),
            "avaliacao_reclamacao": ("Classificação do consumidor sobre o desfecho da reclamação.", "bronze.Avaliação Reclamação + trim."),
            "nota_consumidor": ("Nota de 1 a 5 dada pelo consumidor; nula se a reclamação não foi avaliada.", "bronze.Nota do Consumidor + trim + try_cast para INT."),
            "interacao_judiciario": ("Indicação publicada só no arquivo de 2025-09; nula nos demais meses.",
                                     "bronze.Interação com Judiciario: S → verdadeiro, N → falso."),
            "data_ultimo_complemento": ("Data do último complemento do consumidor, publicada só no arquivo de 2025-09; nula nos demais meses.",
                                        "bronze.Último Complemento Consumidor + trim, convertida com try_to_date no formato dd/mm/aaaa."),
            "linha_repetida": ("Verdadeiro quando existe outra linha com as 21 colunas da fonte idênticas. Sem ID na fonte, pode ser duplicata ou reclamação real com os mesmos atributos.",
                               "Calculada na Silver: contagem por janela sobre as 21 colunas da Bronze maior que 1 (notebooks/03, H9)."),
            "_arquivo_origem": ("Coluna de controle: arquivo de origem da linha.", "Copiada de bronze._arquivo_origem."),
            "_data_ingestao": ("Coluna de controle: momento da carga na Bronze.", "Copiada de bronze._data_ingestao."),
        },
    },
    "mvp_reclamacoes.gold.fato_reclamacao": {
        "descricao": (
            f"Fato do esquema estrela: uma linha por reclamação finalizada contra empresas do recorte bancário ({RECORTE}), "
            "de jan/2021 a ago/2026. Origem: mvp_reclamacoes.silver.reclamacoes filtrada pelo recorte. Transformação: notebooks/05_gold."
        ),
        "colunas": {
            "sk_tempo": ("Chave da dim_tempo: data de finalização como inteiro aaaammdd.", "silver.data_finalizacao formatada como aaaammdd."),
            "sk_assunto": ("Chave da dim_assunto.", "xxhash64 de silver.assunto (texto)."),
            "sk_problema": ("Chave da dim_problema.", "xxhash64 de silver.problema (texto)."),
            "sk_empresa": ("Chave da dim_empresa.", "xxhash64 de silver.nome_empresa (texto)."),
            "sk_local": ("Chave da dim_local.", "xxhash64 de silver.uf (texto)."),
            "sk_perfil": ("Chave da dim_perfil.", "xxhash64 do sexo com rótulo, de silver.faixa_etaria e de silver.canal_contratacao (texto)."),
            "tempo_resposta_dias": ("Dias até a resposta da empresa; nulo se não respondida ou se o valor publicado era inválido.", "silver.tempo_resposta_dias, sem alteração."),
            "nota_consumidor": ("Nota de 1 a 5 dada pelo consumidor; nula se não avaliada.", "silver.nota_consumidor, sem alteração."),
            "foi_respondida": ("Se a empresa respondeu a reclamação.", "silver.respondida, sem alteração."),
            "foi_avaliada": ("Se o consumidor avaliou a reclamação (tem nota).", "silver.nota_consumidor não nula (notebooks/03, H7: nota vazia coincide com Não Avaliada)."),
            "foi_resolvida": ("Verdadeiro se avaliada como Resolvida, falso se Não Resolvida, nulo se não avaliada ou se o resultado está ausente.",
                              "silver.avaliacao_reclamacao: Resolvida → verdadeiro, Não Resolvida → falso, demais → nulo."),
            "linha_repetida": ("Se existe outra linha com as 21 colunas da fonte idênticas (permite medir a sensibilidade da análise).", "silver.linha_repetida, sem alteração."),
        },
    },
    "mvp_reclamacoes.gold.dim_tempo": {
        "descricao": "Dimensão de tempo: calendário diário completo de 01/01/2021 a 31/08/2026. Transformação: notebooks/05_gold.",
        "colunas": {
            "sk_tempo": ("Chave da dimensão: data como inteiro aaaammdd.", "Gerada a partir da data."),
            "data": ("Data do calendário.", "Gerada com sequence de 2021-01-01 a 2026-08-31; não vem da fonte."),
            "ano": ("Ano da data.", "year(data)."),
            "mes": ("Mês da data (1 a 12).", "month(data)."),
            "ano_mes": ("Ano e mês no formato aaaa-mm.", "date_format(data, yyyy-MM)."),
        },
    },
    "mvp_reclamacoes.gold.dim_assunto": {
        "descricao": ("Dimensão de assunto (produto ou serviço reclamado) das reclamações do recorte, com a área. "
                      "Cada assunto pertence a uma única área (notebooks/03, H1). Transformação: notebooks/05_gold."),
        "colunas": {
            "sk_assunto": ("Chave da dimensão.", "xxhash64 de assunto (texto)."),
            "area": ("Área à qual pertence o assunto.", "silver.area."),
            "assunto": ("Assunto (produto ou serviço) da reclamação.", "silver.assunto."),
        },
    },
    "mvp_reclamacoes.gold.dim_problema": {
        "descricao": ("Dimensão de problema das reclamações do recorte, com o grupo. Cada problema pertence a um único grupo "
                      "(notebooks/03, H1). Transformação: notebooks/05_gold."),
        "colunas": {
            "sk_problema": ("Chave da dimensão.", "xxhash64 de problema (texto)."),
            "grupo_problema": ("Agrupamento do problema.", "silver.grupo_problema."),
            "problema": ("Problema padronizado (com a de-para da Silver).", "silver.problema."),
        },
    },
    "mvp_reclamacoes.gold.dim_empresa": {
        "descricao": ("Dimensão de empresa do recorte, com nome padronizado e segmento. Dentro do recorte, cada empresa tem um "
                      "único segmento (verificado em notebooks/04_silver). Transformação: notebooks/05_gold."),
        "colunas": {
            "sk_empresa": ("Chave da dimensão.", "xxhash64 de nome_empresa (texto)."),
            "nome_empresa": ("Nome padronizado da empresa.", "silver.nome_empresa."),
            "segmento_mercado": ("Segmento de mercado da empresa.", "silver.segmento_mercado."),
        },
    },
    "mvp_reclamacoes.gold.dim_local": {
        "descricao": ("Dimensão de local do consumidor no grão de UF, com a região. Cada UF pertence a uma única região "
                      "(notebooks/03, H5). A cidade fica só na Silver. Transformação: notebooks/05_gold."),
        "colunas": {
            "sk_local": ("Chave da dimensão.", "xxhash64 de uf (texto)."),
            "uf": ("Sigla do estado do consumidor.", "silver.uf."),
            "regiao": ("Sigla da região do consumidor.", "silver.regiao."),
        },
    },
    "mvp_reclamacoes.gold.dim_perfil": {
        "descricao": ("Dimensão de perfil (junk dimension): combinações de sexo, faixa etária e canal de contratação presentes "
                      "no recorte. Transformação: notebooks/05_gold."),
        "colunas": {
            "sk_perfil": ("Chave da dimensão.", "xxhash64 de sexo, faixa_etaria e canal_contratacao (texto)."),
            "sexo": ("Sexo do consumidor. O = Outro por definição do projeto (o dicionário v1.1 da fonte documenta só F e M); Não informado quando a fonte veio vazia.",
                     "silver.sexo: F → Feminino, M → Masculino, O → Outro, nulo → Não informado."),
            "faixa_etaria": ("Faixa etária do consumidor.", "silver.faixa_etaria."),
            "canal_contratacao": ("Meio de contratação ou aquisição do produto ou serviço.", "silver.canal_contratacao."),
        },
    },
    "mvp_reclamacoes.gold.agg_reclamacoes_mensais": {
        "descricao": ("Agregado mensal: total de reclamações finalizadas na plataforma inteira e no recorte bancário. "
                      "Base da participação do setor na pergunta P1. Origem: silver.reclamacoes. Transformação: notebooks/05_gold."),
        "colunas": {
            "ano_mes": ("Mês de finalização no formato aaaa-mm.", "date_format(silver.data_finalizacao, yyyy-MM)."),
            "total_plataforma": ("Reclamações finalizadas no mês, em todos os setores.", "count(*) de silver.reclamacoes por mês."),
            "total_recorte": (f"Reclamações finalizadas no mês no recorte bancário ({RECORTE}).", "Soma, por mês, das linhas de silver.reclamacoes no recorte."),
        },
    },
}

## 2. Conferência: o catálogo cobre exatamente as colunas reais

Para cada tabela, as colunas descritas acima precisam ser **as mesmas** do esquema real, sem faltar nem sobrar nenhuma. Se alguma diferir, o notebook para antes de gravar qualquer comentário.

In [ ]:
for tabela, info in CATALOGO.items():
    reais = spark.table(tabela).columns
    descritas = list(info["colunas"])
    faltando, sobrando = sorted(set(reais) - set(descritas)), sorted(set(descritas) - set(reais))
    print(f"{tabela}: {len(reais)} colunas reais, {len(descritas)} descritas | faltando={faltando} sobrando={sobrando}")
    assert not faltando and not sobrando, f"{tabela}: o catálogo não bate com o esquema real"
print("\nO catálogo cobre exatamente as colunas das 10 tabelas.")

## 3. Domínio medido e gravação dos comentários

Para cada coluna, o domínio é calculado nos dados atuais:
- **números e datas:** mínimo e máximo;
- **booleanos:** contagem de verdadeiros e falsos;
- **chaves `sk_*`:** total de valores distintos;
- **texto:** a lista de valores quando há até 12 distintos, ou o total de distintos quando há mais;
- **todas as colunas:** a contagem de nulos.

In [ ]:
LIMITE_LISTA = 12


def numero(n):
    return f"{n:,}".replace(",", ".")


def literal_sql(texto):
    return "'" + texto.replace("\\", "\\\\").replace("'", "\\'") + "'"


def medir_dominios(tabela):
    df = spark.table(tabela)
    tipos = dict(df.dtypes)
    agregacoes = []
    for coluna, tipo in df.dtypes:
        agregacoes.append(F.sum(F.col(coluna).isNull().cast("int")).alias(f"{coluna}|nulos"))
        if tipo in ("int", "bigint", "date", "timestamp") and not coluna.startswith("sk_"):
            agregacoes += [F.min(coluna).alias(f"{coluna}|min"), F.max(coluna).alias(f"{coluna}|max")]
        if tipo == "boolean":
            agregacoes += [F.sum(F.col(coluna).cast("int")).alias(f"{coluna}|verdadeiros"),
                           F.sum((~F.col(coluna)).cast("int")).alias(f"{coluna}|falsos")]
    r = df.agg(*agregacoes).first().asDict()

    dominios = {}
    for coluna, tipo in tipos.items():
        nulos = numero(r[f"{coluna}|nulos"] or 0)
        if coluna.startswith("sk_"):
            texto = f"chave substituta com {numero(df.select(coluna).distinct().count())} valores distintos"
        elif tipo == "boolean":
            texto = f"verdadeiro {numero(r[f'{coluna}|verdadeiros'] or 0)}; falso {numero(r[f'{coluna}|falsos'] or 0)}"
        elif tipo in ("int", "bigint", "date", "timestamp"):
            texto = f"mínimo {r[f'{coluna}|min']}, máximo {r[f'{coluna}|max']}"
        else:
            distintos = df.select(coluna).dropna().distinct()
            n = distintos.count()
            if n <= LIMITE_LISTA:
                valores = sorted(linha[0] for linha in distintos.collect())
                texto = "valores " + ", ".join(f'"{v}"' for v in valores)
            else:
                texto = f"{numero(n)} valores distintos"
        dominios[coluna] = f"{texto}; nulos {nulos}"
    return dominios


for tabela, info in CATALOGO.items():
    dominios = medir_dominios(tabela)
    spark.sql(f"COMMENT ON TABLE {tabela} IS {literal_sql(info['descricao'])}")
    for coluna, (descricao, linhagem) in info["colunas"].items():
        comentario = f"{descricao} Domínio: {dominios[coluna]}. Linhagem: {linhagem}"
        spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN `{coluna}` COMMENT {literal_sql(comentario)}")
    print(f"comentários gravados: {tabela} ({len(info['colunas'])} colunas)")

## 4. Catálogo em Markdown para o README

Lê os comentários **de volta do Unity Catalog** (`information_schema`). Assim, o que for transcrito no README é exatamente o que está gravado.

In [ ]:
colunas_uc = spark.sql("""
    SELECT table_schema, table_name, column_name, full_data_type, comment, ordinal_position
    FROM mvp_reclamacoes.information_schema.columns
    WHERE table_schema IN ('bronze', 'silver', 'gold')
""").collect()
tabelas_uc = {(t.table_schema, t.table_name): t.comment for t in spark.sql("""
    SELECT table_schema, table_name, comment
    FROM mvp_reclamacoes.information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold') AND table_type <> 'VIEW'
""").collect()}

for tabela in CATALOGO:
    _, schema, nome = tabela.split(".")
    print(f"\n### {schema}.{nome}\n\n{tabelas_uc[(schema, nome)]}\n")
    print("| Coluna | Tipo | Descrição, domínio e linhagem |")
    print("|---|---|---|")
    linhas = sorted((c for c in colunas_uc if c.table_schema == schema and c.table_name == nome), key=lambda c: c.ordinal_position)
    for c in linhas:
        print(f"| `{c.column_name}` | {c.full_data_type} | {(c.comment or '').replace('|', '/')} |")